# 10 - Lalonde / NSW job training benchmark

Every notebook so far used synthetic data, where the true effect is known and
the estimators recover it. This one uses the dataset that made the field
sceptical of exactly that reassurance.

The National Supported Work demonstration randomised job training, so the
experimental effect on 1978 earnings is known — roughly **$1,800**. LaLonde
(1986) then asked whether observational methods, applied to the same treated
group but with a non-experimental comparison group drawn from a national survey,
could recover that number. Largely they could not, and that finding shaped how
the field treats observational estimates.

This notebook runs our estimators on that setup. It is the one notebook here
whose honest conclusion is that the analysis does not support a causal claim.

## Causal question

What was the effect of the job training programme on participants' 1978
earnings?

## Data and design

- **Unit of analysis:** one individual.
- **Treatment:** `treatment` — 185 men who received training, from the
  randomised demonstration.
- **Comparison group:** 429 men drawn from the PSID survey, who were never part
  of the experiment.
- **Outcome:** `outcome`, 1978 earnings in dollars.
- **Covariates:** age, education, ethnicity indicators, marital status, degree
  status, and 1974/1975 earnings with their zero-earnings flags.

The design is the point. The treated group is experimental; the comparison group
is not. Any difference between them reflects both the programme and whatever
distinguishes trainees from a national survey sample.

Run `python scripts/prepare_lalonde_job_training_dataset.py` first if the file
below is missing.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import warnings

import numpy as np
import pandas as pd

from causal_inference_lab.diagnostics import balance_table, ipw_weights
from causal_inference_lab.estimators import aipw_ate, difference_in_means, g_computation_ate, ipw_ate
from causal_inference_lab.matching import nearest_neighbour_matching, propensity_score_matching
from causal_inference_lab.uncertainty import bootstrap_ate

# The propensity model is fitted on unscaled earnings in the tens of thousands,
# which lbfgs does not converge on. That is itself a finding -- see Diagnostics.
warnings.filterwarnings("ignore", message=".*lbfgs failed to converge.*")

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "lalonde_job_training.csv"
if not DATA_PATH.exists():
    sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
    from prepare_lalonde_job_training_dataset import prepare_lalonde_job_training_dataset

    prepare_lalonde_job_training_dataset()

data = pd.read_csv(DATA_PATH)
COVARIATES = [
    "age", "education", "black", "hispanic", "married",
    "nodegree", "earnings_74", "earnings_75", "u74", "u75",
]
EXPERIMENTAL_BENCHMARK = 1_800.0

print(f"observations:  {len(data):,}")
print(f"treated:       {int(data['treatment'].sum())}")
print(f"comparison:    {int(len(data) - data['treatment'].sum())}")
print()
print("mean characteristics by group:")
print(data.groupby("treatment")[["age", "education", "black", "married", "earnings_74", "earnings_75"]]
      .mean().to_string(float_format=lambda v: f"{v:,.2f}"))

**Interpretation.** The two groups are not remotely comparable. The comparison
group earned far more in 1974 and 1975, is more likely to be married, and
differs sharply in ethnic composition. These are not subtle imbalances to be
tidied up by adjustment — they are the signature of two populations sampled in
completely different ways.

## Estimand

The **average treatment effect on the treated (ATT)**: the effect of training on
the men who received it.

The ATT is the relevant quantity — nobody proposes giving this programme to a
random national sample — and it is also what the experimental benchmark
measures, which makes the comparison meaningful.

## Identification assumptions

1. **Conditional ignorability.** Given the covariates, participation is as good
   as random. This is the claim LaLonde put to the test, and the one that fails.
2. **Overlap.** Every treated man has comparable untreated men. Checked below,
   and it does not hold well.
3. **Consistency and no interference.**

There is a fourth, unstated assumption in any such analysis: that the measured
covariates capture why these two groups differ. Since one group was recruited
into a training demonstration and the other was sampled by a national survey,
that is a strong claim.

## Estimation

Every estimator in the repository, on the same data, against a known
experimental benchmark.

In [ ]:
matched_nn = nearest_neighbour_matching(data, covariates=COVARIATES)
matched_ps = propensity_score_matching(data, covariates=COVARIATES)

results = pd.DataFrame(
    [
        ("naive difference in means", difference_in_means(data).estimate, "ATE"),
        ("g-computation", g_computation_ate(data, covariates=COVARIATES).estimate, "ATE"),
        ("IPW", ipw_ate(data, covariates=COVARIATES).estimate, "ATE"),
        ("AIPW", aipw_ate(data, covariates=COVARIATES).estimate, "ATE"),
        ("nearest-neighbour matching", matched_nn.effect.estimate, "ATT"),
        ("propensity score matching", matched_ps.effect.estimate, "ATT"),
    ],
    columns=["method", "estimate", "estimand"],
)
results["error vs benchmark"] = results["estimate"] - EXPERIMENTAL_BENCHMARK

print(f"experimental benchmark: ${EXPERIMENTAL_BENCHMARK:,.0f}\n")
print(results.to_string(index=False, float_format=lambda v: f"{v:,.0f}"))
print(f"\nspread across methods: ${results['estimate'].max() - results['estimate'].min():,.0f}")

**Interpretation.** Not one estimator recovers the benchmark, and they do not
even agree on the sign.

The naive comparison says the programme *reduced* earnings by $635 — an artefact
of comparing trainees against a better-off survey sample. Adjustment does not
rescue it: g-computation gives +$866, IPW gives −$1,478, AIPW gives +$78, and
matching gives roughly $0 and −$1,206. The spread is about $2,300, wider than
the effect being estimated.

Had this been a real analysis with no benchmark to check against, any of these
numbers could have been reported with a straight face — and a method could have
been chosen, after the fact, to support whatever conclusion was wanted. That is
the LaLonde critique in one table.

## Diagnostics

The estimates disagree, so the diagnostics should tell us why. Balance first.

In [ ]:
before = balance_table(data, covariates=COVARIATES)
weights = ipw_weights(data, covariates=COVARIATES)
after = balance_table(data, covariates=COVARIATES, weights=weights)

comparison = before[["covariate", "smd"]].merge(
    after[["covariate", "smd"]], on="covariate", suffixes=("_before", "_after")
)
print(comparison.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print(f"\nworst |SMD| before weighting: {before['abs_smd'].max():.3f}")
print(f"worst |SMD| after weighting:  {after['abs_smd'].max():.3f}")
print(f"covariates still above the 0.1 threshold: "
      f"{int((after['abs_smd'] > 0.1).sum())} of {len(after)}")

**Interpretation.** Before weighting the imbalance is extreme: 1.67 standard
deviations on `black` and 1.00 on `u74`, where 0.1 is the conventional limit for
a usable comparison.

Weighting improves matters a great deal but does not finish the job. The worst
imbalance is still 0.226, and 8 of 10 covariates remain above 0.1. In notebook
01, the same procedure on synthetic data brought every covariate under 0.031.
Here the propensity model cannot construct a comparison group that resembles the
trainees, because within this comparison pool no such group exists.

Balance failing is usually a symptom of an overlap failure. The weights say so
directly.

In [ ]:
print(f"largest weight:              {weights.max():.1f}")
print(f"observations:                {len(weights)}")
print(f"weight share of heaviest unit: {weights.max() / weights.sum():.1%}")
print(f"weight share of heaviest 5%:   "
      f"{np.sort(weights)[-len(weights) // 20:].sum() / weights.sum():.1%}")

effective_n = weights.sum() ** 2 / (weights**2).sum()
print(f"\neffective sample size: {effective_n:.0f} of {len(weights)}")

**Interpretation.** A single observation carries a weight of 48 in a dataset of
614 — 4.3% of all the weight resting on one person. The heaviest 5% of units
hold a quarter of it, and the effective sample size is **200**, not 614.

This is what an overlap violation looks like in practice. The IPW estimate rests
on a handful of survey respondents who happen to resemble trainees, and its
value is largely determined by their individual outcomes. The estimate is not
so much wrong as unsupported — there is very little data behind it.

A related detail from the setup: the propensity model does not converge on these
covariates, because earnings enter unscaled in the tens of thousands. We
suppressed that warning to keep the output readable, which is exactly the kind of
convenience worth flagging rather than hiding.

## Uncertainty

Given all of the above, the interval matters more than the point estimate.

In [ ]:
interval = bootstrap_ate(
    data,
    estimator=aipw_ate,
    covariates=COVARIATES,
    n_bootstrap_samples=300,
    seed=1,
)

print(f"AIPW estimate: ${interval.estimate:,.0f}")
print(f"95% interval:  [${interval.lower:,.0f}, ${interval.upper:,.0f}]")
print(f"standard error: ${interval.std_error:,.0f}")
print()
print(f"benchmark ${EXPERIMENTAL_BENCHMARK:,.0f} inside the interval: "
      f"{interval.lower <= EXPERIMENTAL_BENCHMARK <= interval.upper}")
print(f"zero inside the interval:              "
      f"{interval.lower <= 0 <= interval.upper}")
print(f"\ninterval width: ${interval.upper - interval.lower:,.0f}")

**Interpretation.** The interval runs from about −$1,800 to +$1,800. It contains
the experimental benchmark, it contains zero, and it contains a substantial
negative effect. Its width is roughly twice the effect we are trying to detect.

So the correct summary of this analysis is not "the effect is $78". It is that
these data, with these methods, cannot distinguish a beneficial programme from a
harmful one. The interval covering the benchmark is not a success — an interval
that wide would cover almost any plausible answer.

## Limitations

- **Conditional ignorability is not credible here.** The comparison group was
  sampled from a national survey; the treated group volunteered for a training
  demonstration. The measured covariates do not capture that difference, which
  is the entire finding.
- **Overlap fails.** A weight of 48 in 614 observations, an effective sample
  size of 200, and 8 of 10 covariates still imbalanced after weighting: there is
  no comparable comparison group to be found in this pool.
- **Estimates are not comparable across rows.** Four target the ATE and two the
  ATT. On these data that distinction is minor next to the specification
  problem, but it is real.
- **The benchmark is approximate.** Published experimental estimates for this
  demonstration vary by subsample and specification; $1,800 is a round figure,
  not a precise target.
- **The propensity model does not converge.** Earnings are unscaled. Scaling
  would fix convergence and would not fix overlap.
- **A negative result is the finding.** Nothing here should be read as "the
  programme did not work" — the randomised evidence says it did. It should be
  read as: these observational methods, on this comparison group, could not have
  told us so.